In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

In [3]:
# Initialize Faker with for realistic names/cities
fake = Faker('en_IN')
Faker.seed(42) # Ensures we get the same random data 
np.random.seed(42)

# Setting the scale
NUM_CUSTOMERS = 2000
NUM_RESELLERS = 100
NUM_PRODUCTS = 500
NUM_ORDERS = 10000

print("Generating Customers...")

Generating Customers...


In [4]:
# --- 1. CUSTOMERS TABLE ---
customer_data = []
for _ in range(NUM_CUSTOMERS):
    # Simulating the Meesho demographic: heavy on Tier 2/3 cities
    tier = np.random.choice(['Tier 1', 'Tier 2', 'Tier 3'], p=[0.2, 0.5, 0.3])
    customer_data.append({
        'customer_id': fake.unique.random_int(min=10000, max=99999),
        'city': fake.city(),
        'city_tier': tier,
        'signup_date': fake.date_between(start_date='-2y', end_date='today'),
        'gender': np.random.choice(['F', 'M'], p=[0.65, 0.35]) # Reseller apps lean towards females
    })
customers_df = pd.DataFrame(customer_data)


print("Generating Resellers & Products...")

Generating Resellers & Products...


In [5]:
# --- 2. RESELLERS TABLE ---
reseller_data = []
categories = ['Apparel', 'Home & Kitchen', 'Electronics', 'Beauty', 'Jewelry']
for _ in range(NUM_RESELLERS):
    reseller_data.append({
        'reseller_id': fake.unique.random_int(min=100, max=999),
        'category_focus': np.random.choice(categories),
        'city': fake.city(),
        'join_date': fake.date_between(start_date='-3y', end_date='today')
    })
resellers_df = pd.DataFrame(reseller_data)

In [6]:
# --- 3. PRODUCTS TABLE ---
product_data = []
for _ in range(NUM_PRODUCTS):
    cat = np.random.choice(categories)
    # Assigning random reseller 
    r_id = np.random.choice(resellers_df['reseller_id'])
    
    # Prices varying by categories
    if cat == 'Electronics':
        price = round(random.uniform(500, 5000), 2)
    elif cat == 'Apparel':
        price = round(random.uniform(200, 1500), 2)
    else:
        price = round(random.uniform(100, 1000), 2)
        
    product_data.append({
        'product_id': fake.unique.random_int(min=1000, max=9999),
        'category': cat,
        'base_price': price,
        'reseller_id': r_id
    })
products_df = pd.DataFrame(product_data)


print("Generating Orders (This is the core business logic)...")

Generating Orders (This is the core business logic)...


In [7]:
# --- 4. ORDERS TABLE ---
order_data = []
for _ in range(NUM_ORDERS):
    c_id = np.random.choice(customers_df['customer_id'])
    p_id = np.random.choice(products_df['product_id'])
    
    #product price to calculate final order value
    base_price = products_df.loc[products_df['product_id'] == p_id, 'base_price'].values[0]
    
    #customer city tier to influence payment method
    c_tier = customers_df.loc[customers_df['customer_id'] == c_id, 'city_tier'].values[0]
    
    # Business Logic: Tier 3 uses more COD. Tier 1 uses more UPI.
    if c_tier == 'Tier 3':
        pay_method = np.random.choice(['COD', 'Paytm_UPI', 'Paytm_Wallet'], p=[0.70, 0.25, 0.05])
    elif c_tier == 'Tier 2':
        pay_method = np.random.choice(['COD', 'Paytm_UPI', 'Paytm_Wallet'], p=[0.50, 0.40, 0.10])
    else:
        pay_method = np.random.choice(['COD', 'Paytm_UPI', 'Paytm_Wallet'], p=[0.20, 0.65, 0.15])
        
    # Simulating Discounts (0%, 10%, or 20%)
    discount = np.random.choice([0, 10, 20], p=[0.5, 0.3, 0.2])
    order_value = round(base_price * (1 - (discount / 100)), 2)
    
    # Business Logic: Higher discounts or COD slightly increase return probability
    return_prob = 0.05
    if discount == 20: return_prob += 0.04
    if pay_method == 'COD': return_prob += 0.03
    
    is_returned = np.random.choice([1, 0], p=[return_prob, 1 - return_prob])
    status = 'Returned' if is_returned else 'Delivered'
    
    order_data.append({
        'order_id': fake.unique.random_int(min=100000, max=999999),
        'customer_id': c_id,
        'product_id': p_id,
        'order_date': fake.date_between(start_date='-1y', end_date='today'),
        'quantity': np.random.choice([1, 2, 3], p=[0.8, 0.15, 0.05]),
        'discount_pct': discount,
        'payment_method': pay_method,
        'order_value': order_value,
        'delivery_status': status,
        'return_flag': is_returned
    })
orders_df = pd.DataFrame(order_data)


print("Generating Payment Events...")

Generating Payment Events...


In [8]:
# --- 5. PAYMENT EVENTS TABLE ---
# We only track digital payments here (UPI/Wallet) + successful COD deliveries
payment_data = []
for index, row in orders_df.iterrows():
    # If COD and returned, no payment event happened.
    if row['payment_method'] == 'COD' and row['return_flag'] == 1:
        continue
        
    # Simulate payment failures for digital methods (approx 10% failure rate)
    if row['payment_method'] in ['Paytm_UPI', 'Paytm_Wallet']:
        pay_status = np.random.choice(['Success', 'Failed'], p=[0.90, 0.10])
    else:
        pay_status = 'Success' # COD that didn't return is successful
        
    # Simulate Paytm Cashback (only for digital payments)
    cashback = 1 if (row['payment_method'] in ['Paytm_UPI', 'Paytm_Wallet'] and pay_status == 'Success' and random.random() < 0.3) else 0

    payment_data.append({
        'payment_id': fake.unique.random_int(min=1000000, max=9999999),
        'order_id': row['order_id'],
        'payment_status': pay_status,
        'cashback_applied': cashback
    })
payments_df = pd.DataFrame(payment_data)

In [9]:
# --- SAVE TO CSV ---
print("Saving files to your current folder...")
customers_df.to_csv('customers.csv', index=False)
resellers_df.to_csv('resellers.csv', index=False)
products_df.to_csv('products.csv', index=False)
orders_df.to_csv('orders.csv', index=False)
payments_df.to_csv('payment_events.csv', index=False)

print("Success! You now have 5 CSV files ready for the database.")

Saving files to your current folder...
Success! You now have 5 CSV files ready for the database.
